# matmul-back-transpose-pair — worked example 3: Backward Through Batched Matmul Using transpose(-1,-2)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-back-transpose-pair`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For batched matmul `out = x @ y` with `x: (B, m, k)` and `y: (B, k, n)`, the backward gradients follow the same transpose-pair pattern as 2-D matmul, but the transpose must swap only the last two dimensions. Using `.transpose(-1, -2)` instead of `.T` preserves the batch axis and swaps only the inner matrix dimensions, which is essential for correctness.

## Worked solution

**Step 1 — establish the shapes.**
`x: (B, m, k)`, `y: (B, k, n)`, `out: (B, m, n)`, `grad_out: (B, m, n)`.

**Step 2 — compute dL/dx.**
We need shape `(B, m, k)`. The batched matmul `grad_out @ y.transpose(-1,-2)` is `(B, m, n) @ (B, n, k) = (B, m, k)`. The `.transpose(-1,-2)` swaps `k` and `n` while leaving `B` in place.

**Step 3 — compute dL/dy.**
We need shape `(B, k, n)`. The batched matmul `x.transpose(-1,-2) @ grad_out` is `(B, k, m) @ (B, m, n) = (B, k, n)`.

**Step 4 — why not `.T`?**
On a 3-D tensor `(B, k, n)`, `.T` reverses all axes, giving `(n, k, B)` — wrong batch position and wrong shape. Only `transpose(-1,-2)` is correct. We verify both gradients against autograd.

In [ ]:
import torch as t

t.manual_seed(41)
B, m, k, n = 3, 4, 5, 6

x = t.randn(B, m, k)
y = t.randn(B, k, n)
out = x @ y  # (B, m, n)
grad_out = t.randn_like(out)

# Backward with transpose(-1, -2) — only swaps last two dims
dL_dx = grad_out @ y.transpose(-1, -2)          # (B,m,n)@(B,n,k) = (B,m,k)
dL_dy = x.transpose(-1, -2) @ grad_out          # (B,k,m)@(B,m,n) = (B,k,n)

print(f"x: {x.shape}, y: {y.shape}, out: {out.shape}")
print(f"dL/dx shape: {dL_dx.shape}  expected ({B},{m},{k})")
print(f"dL/dy shape: {dL_dy.shape}  expected ({B},{k},{n})")

# Show why .T is wrong for 3-D
print(f"\ny.T shape (WRONG):              {y.T.shape}")
print(f"y.transpose(-1,-2) shape (OK): {y.transpose(-1,-2).shape}")

# Verify with autograd
x_ag = x.clone().requires_grad_(True)
y_ag = y.clone().requires_grad_(True)
(x_ag @ y_ag).backward(grad_out)

print(f"\ndL/dx match: {t.allclose(dL_dx, x_ag.grad, atol=1e-5)}")
print(f"dL/dy match: {t.allclose(dL_dy, y_ag.grad, atol=1e-5)}")